# Experiment 2: Exploratory Data Analysis (EDA) – Loan Approval Prediction

**Dataset:** `ML DATASET(1).csv`  
**Records:** 20,000 loan applications  
**Features:** 38 input columns  
**Target:** `Loan_Status` (Approved / Rejected)

### Objective
Perform exploratory data analysis on the actual loan dataset by examining its structure, data types, missing values, numerical distributions, categorical relationships, correlations, and potential outliers.

In [ ]:
# Google Colab: upload the exact dataset file
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Load the exact uploaded dataset
df = pd.read_csv('ML DATASET(1).csv')

print('Dataset loaded successfully.')
print('Shape:', df.shape)

In [ ]:
# Basic dataset inspection
print('First 5 rows:')
display(df.head())

print('\nShape:', df.shape)

print('\nColumn names:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.to_frame('Data Type'))

print('\nDataset information:')
df.info()

print('\nMissing values:')
missing = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().mean() * 100).round(2)
})
display(missing[missing['Missing_Count'] > 0])

print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
# Target variable: Loan_Status
print('Loan Status counts:')
print(df['Loan_Status'].value_counts())

print('\nLoan Status percentages:')
print((df['Loan_Status'].value_counts(normalize=True) * 100).round(2))

# Create a numeric flag for EDA only: Approved = 1, Rejected = 0
df['Approved_Flag'] = df['Loan_Status'].map({'Approved': 1, 'Rejected': 0})

if df['Approved_Flag'].isnull().any():
    raise ValueError('Unexpected values found in Loan_Status: ' + str(df.loc[df['Approved_Flag'].isnull(), 'Loan_Status'].unique()))

In [ ]:
# Numerical columns present in the actual dataset
num_cols = [
    'Age', 'Years_at_Current_Job', 'Total_Work_Experience',
    'Monthly_Income', 'Annual_Income', 'Other_Income',
    'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI',
    'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance',
    'Credit_Card_Utilization', 'Number_of_Bank_Accounts',
    'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults',
    'Missed_Payments', 'Loan_Amount', 'Loan_Tenure',
    'Interest_Rate', 'Property_Value', 'Collateral_Value', 'Loan_to_Value'
]

# Categorical columns present in the actual dataset
cat_cols = [
    'Gender', 'Marital_Status', 'Education', 'Residence_Type',
    'Employment_Type', 'Tax_Return_Filed', 'PAN_Verified',
    'Aadhaar_Verified', 'Loan_Purpose', 'Collateral'
]

print('Numerical columns:', len(num_cols))
print(num_cols)
print('\nCategorical columns:', len(cat_cols))
print(cat_cols)

In [ ]:
# Missingness analysis
missing_cols = df.isnull().sum()
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
missing_cols.plot(kind='bar')
plt.title('Missing Values by Column')
plt.xlabel('Columns')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

print('Actual columns containing missing values:')
display(pd.DataFrame({
    'Missing_Count': missing_cols,
    'Missing_Percentage': (missing_cols / len(df) * 100).round(2)
}))

In [ ]:
# Distributions of important numerical variables
plot_cols = [c for c in ['Age', 'Monthly_Income', 'Loan_Amount', 'Credit_Score', 'Debt_to_Income', 'Interest_Rate'] if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.histplot(df[col].dropna(), bins=30, kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)

for j in range(len(plot_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Loan approval rate by employment type
approval_rate = (
    df.groupby('Employment_Type', observed=True)['Approved_Flag']
      .mean().mul(100).sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(x=approval_rate.index, y=approval_rate.values)
plt.title('Loan Approval Rate (%) by Employment Type')
plt.xlabel('Employment Type')
plt.ylabel('Approval Rate (%)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

display(approval_rate.round(2).to_frame('Approval Rate (%)'))

In [ ]:
# Approval rate by selected categorical variables
selected_cat_cols = [
    'Gender', 'Marital_Status', 'Education', 'Residence_Type',
    'Loan_Purpose', 'Collateral', 'Tax_Return_Filed'
]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(selected_cat_cols):
    rate = df.groupby(col, observed=True)['Approved_Flag'].mean().mul(100)
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=axes[i])
    axes[i].set_title(f'Approval Rate by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Approval Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)

for j in range(len(selected_cat_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical features
numeric_df = df.select_dtypes(include=np.number).drop(columns=['Approved_Flag'], errors='ignore')
corr_matrix = numeric_df.corr()

plt.figure(figsize=(18, 14))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for important numerical variables to identify possible outliers
box_cols = [c for c in ['Age', 'Monthly_Income', 'Loan_Amount', 'Credit_Score', 'Debt_to_Income', 'Interest_Rate'] if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(box_cols):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(f'Boxplot of {col}')

for j in range(len(box_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Key EDA Findings

- The dataset contains **20,000 loan applications and 39 columns**.
- `Loan_Status` is the target variable with **Approved** and **Rejected** classes.
- Missing values are present in `Other_Income`, `Investments`, `Bank_Balance`, `Credit_Score`, `Tax_Return_Filed`, `Interest_Rate`, `Property_Value`, `Collateral_Value`, and `Loan_to_Value`.
- `Application_ID` is an identifier and should not be treated as a predictive numerical feature during model training.
- Numerical distributions, categorical approval rates, correlations, and boxplots are examined before preprocessing and model development.